# 📋 Phase 3: Results Compilation

The original 36-trial matrix in this slot duplicated models 1-6, already trained
individually (with bug fixes: normalization, output-bias init, NaN guards,
`clipnorm`, SVM subsampling, epoch-level resume) in notebooks `03`-`08`.
Re-running it here would redo ~24-36 trials with the OLD, unfixed model code.

This notebook only compiles the final leaderboard from the results you already
have in `results.csv`, once all of `03`-`08` have finished.

## 0. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pandas as pd

OUTPUT_DIR = "/content/drive/MyDrive/epilepsy_outputs"
RESULTS    = os.path.join(OUTPUT_DIR, "results.csv")

In [ ]:
# ── Leaderboard ────────────────────────────────────────────────────────────────────
df_res = pd.read_csv(RESULTS)

expected_models = {"CNN2D", "conformer", "lstm", "cnn_lstm", "lstm_conformer", "complex_hybrid"}
found_models = set(df_res["model"].unique())
missing = expected_models - found_models
if missing:
    print(f"⚠️  Missing results for: {sorted(missing)} -- make sure those notebooks have finished.\n")

expected_rows_per_model = 6  # 2 datasets x 3 optimizers
counts = df_res.groupby("model").size()
incomplete = counts[counts < expected_rows_per_model]
if len(incomplete):
    print("⚠️  Models with fewer than 6 trials logged (still running or interrupted):")
    print(incomplete.to_string(), "\n")

df_sorted = df_res.sort_values("best_val_auc", ascending=False)

cols = [c for c in ["model", "dataset", "optimizer", "best_val_auc", "accuracy",
                     "sensitivity", "specificity", "f1_macro", "f1_weighted",
                     "train_time_min"] if c in df_sorted.columns]

print("=" * 90)
print("  FULL LEADERBOARD — sorted by best_val_auc (descending)")
print("=" * 90)
print(df_sorted[cols].to_string(index=False))

print("\n" + "=" * 90)
print("  BEST TRIAL PER MODEL (by best_val_auc)")
print("=" * 90)
best_per_model = df_sorted.loc[df_sorted.groupby("model")["best_val_auc"].idxmax()]
print(best_per_model[cols].to_string(index=False))

out_path = os.path.join(OUTPUT_DIR, "leaderboard.xlsx")
with pd.ExcelWriter(out_path) as writer:
    df_sorted[cols].to_excel(writer, sheet_name="All_Trials", index=False)
    best_per_model[cols].to_excel(writer, sheet_name="Best_Per_Model", index=False)
print(f"\n✅ Leaderboard saved to {out_path}")